 Scenario: AI-Assisted Email Workflow (Question-Based)

 Context

 A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
 The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
 and sent or rejected.

 1. State Definition
 The assistant maintains a notebook-like state:
 - task → The subject or purpose of the email (e.g., "Q3 Report").
 - draft → The AI-generated email draft.
 - approved → A flag indicating whether the human reviewer has approved the draft.

 2. Workflow (Graph of States)
 Each email task flows through nodes:
 - Draft Node
 - AI generates a draft email based on the task.
 - Updates draft.
 - Review Node (Interrupt)
 - Execution pauses here.
 - Human reviewer inspects the draft and decides whether to approve or reject.
 - Updates approved.
 - Send Node
 - If approved = True → Email is sent.
 - If approved = False → Email is rejected.
 - Updates task with final status (SENT or REJECTED).
 - End Node
 - Workflow completes.

 3. Example Flow
 - Employee: "Draft an email for the Q3 Report."
 - State: task = "Q3 Report"
 - Assistant drafts:
 Dear User,
 Regarding: Q3 Report
 [AI drafted content]
 - Human reviews → Approves draft (approved = True)
 - Assistant sends → task = "SENT: Q3 Report"
 - Final Output: Email delivered.

 Scenario Question:
 "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
 then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
  and human oversight in the process?"

In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('newapi')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f"Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):
    # Interrupt node: waits for human approval
    print(f"Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pauses here until human updates 'approved'

def send_email(state: EmailState):
    if state.get("approved", False):
        print("Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # Pause before review
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)

Drafting email for task: Q3 Report
Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': "Subject: Q3 Report: Key Highlights and Performance Overview\n\nDear [Recipient's Name],\n\nI am writing to share with you the Q3 report, which summarizes our company's performance for the quarter ending [Quarter End Date]. This report provides a comprehensive overview of our financial results, operational achievements, and strategic outcomes.\n\n**Key Highlights:**\n\n* Revenue growth: Our revenue has increased by [X]% compared to the same period last year, driven by strong demand for our products/services.\n* Operational efficiency: We have achieved a [X]% reduction in operational costs, resulting in improved profitability.\n* Strategic initiatives: Our investments in [initiative name] have yielded encouraging results, with [X]% increase in customer engagement.\n\n**Performance Overview:**\n\n- [Financial Metric]: $[X] vs. $[Previous Quarter/X Quarter Last Year]\n- [Operational Metric]: [X]% vs. [Previous Quarter/X Quarter Last Year]\n- [Strategic Metr

Scenario Question

Imagine you are designing an AI-powered assistant that drafts
structured reports, pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

In [4]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata
from datetime import datetime

# 1. STATE (HITL ENABLED)
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    reviewer: str
    feedback: str
    status: str
    history: List[Dict]



# 2. GROQ CALL
def groq_call(prompt: str):
    api_key = userdata.get('newapi')

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    return response.json()["choices"][0]["message"]["content"]


# 3. NODES
# AI Node → Draft generation
def draft_report(state: ReportState):
    draft = groq_call(f"Generate a structured report for: {state['task']}")
    print("\nDraft Generated\n")
    return {"draft": draft, "status": "drafted"}


# HITL Node → Human Review
def human_review(state: ReportState):
    print("\nHUMAN REVIEW REQUIRED")
    print("\nDraft:\n", state["draft"])
    print("\n➡ Waiting for human approval...")
    return {}  # execution pauses here


# Decision Routing
def approval_check(state: ReportState):
    return "approved" if state["approved"] else "rejected"


# Publish Node
def publish_report(state: ReportState):
    print("\nReport Approved & Published")
    return {"status": "published"}


# Reject Node
def reject_report(state: ReportState):
    print("\nReport Rejected")
    return {"status": "rejected"}


# Logging Node (Accountability)
def log_history(state: ReportState):
    entry = {
        "task": state["task"],
        "status": state["status"],
        "reviewer": state["reviewer"],
        "feedback": state["feedback"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    print("\nAction Logged:", entry)

    return {"history": state["history"] + [entry]}


# 4. GRAPH
graph = StateGraph(ReportState)

graph.add_node("draft", draft_report)
graph.add_node("review", human_review)
graph.add_node("publish", publish_report)
graph.add_node("reject", reject_report)
graph.add_node("log", log_history)

# Flow
graph.set_entry_point("draft")
graph.add_edge("draft", "review")

graph.add_conditional_edges(
    "review",
    approval_check,
    {
        "approved": "publish",
        "rejected": "reject"
    }
)

graph.add_edge("publish", "log")
graph.add_edge("reject", "log")
graph.add_edge("log", END)


# 5. ENABLE HITL
checkpointer = MemorySaver()

app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)


# 6. RUN
thread = {"configurable": {"thread_id": "report-1"}}

# Step 1: AI generates draft (execution pauses before review)
app.invoke({
    "task": "Annual Performance Report",
    "draft": "",
    "approved": False,
    "reviewer": "",
    "feedback": "",
    "status": "",
    "history": []
}, thread)

# Step 2: Human provides input (resume execution)
app.invoke({
    "approved": True,
    "reviewer": "Team Lead",
    "feedback": "Accurate and well-structured"
}, thread)


Draft Generated


Draft Generated



{'task': 'Annual Performance Report',
 'draft': "**Annual Performance Report**\n\n**Introduction**\n\nThis Annual Performance Report provides an overview of our organization's achievements and progress over the past year. The report highlights key accomplishments, areas of improvement, and strategic directions for the future.\n\n**Executive Summary**\n\nIn the past year, our organization has made significant strides in achieving its mission and objectives. Some of the key highlights include:\n\n- **Revenue Growth**: We have experienced a 15% increase in revenue, driven by a rise in sales and an expansion of our product offerings.\n- **Customer Satisfaction**: Our customer satisfaction ratings have improved by 10%, reflecting our commitment to delivering high-quality services and products.\n- **Innovation**: We have launched several new product lines and have invested in research and development, positioning ourselves for future growth and competitiveness.\n- **Employee Engagement**: We